# AP_2026Q2 · laboratorio incremental de modelos

Este notebook es una guía ejecutable para comparar diez familias y documentar quince rondas. El objetivo es clasificar condition (new/used) maximizando accuracy.

## Mapa conceptual

~~~mermaid
flowchart TD
 A[JSONL MercadoLibre] --> B[EDA y limpieza]
 B --> C[features tabulares]
 B --> D[TF-IDF + SVD]
 C --> E[CatBoost / LightGBM / XGBoost]
 D --> E
 E --> F[holdout agrupado]
 F --> G[accuracy y errores]
 G --> H[submission ID 1..30000]
~~~

La separación agrupada evita que una familia de publicación (parent_item_id) aparezca en desarrollo y validación. No se usa condition ni un atributo que la nombre directamente.


## Dataset y señal

- **Tabulares:** vendedor, categoría, tipo de listado, moneda, logística, precio, stock, ventas, fechas y conteos de listas anidadas.
- **Texto:** título y texto legible de atributos. TF-IDF representa n-gramas; SVD lo comprime para que un árbol lo use.
- **Categóricas:** CatBoost calcula estadísticas ordenadas y evita convertir cada vendedor en miles de columnas one-hot.

### Cómo leer un árbol de decisión

~~~mermaid
flowchart LR
 A[precio / categoria] --> B{regla}
 B -->|si| C[segmento nuevo]
 B -->|no| D[otra regla]
 D --> E[prediccion usada]
~~~

Un árbol individual tiene alta varianza; boosting suma árboles pequeños que corrigen los errores del conjunto anterior.


## Modelo 1 — CatBoost enriquecido (ganador)

**Funcionamiento.** Construye árboles secuencialmente y calcula estadísticas de categorías con orden aleatorio para reducir target leakage. Aquí aprende combinaciones como seller_id × category_id, logística, stock/ventas y atributos.

**Hiperparámetros clave.** iterations (cantidad de árboles), depth (complejidad), learning_rate (tamaño de cada corrección), l2_leaf_reg (regularización), cat_features (columnas categóricas), early_stopping_rounds.

**Ventajas:** excelente para categorías de alta cardinalidad, poca preparación, interacciones no lineales. **Desventajas:** más lento y pesado; puede sobreajustar si la validación no representa el test.

**Score:** 0,927619 en la ronda 9 con SVD + 75 atributos + interacciones.


## Modelo 2 — XGBoost y Modelo 3 — LightGBM

Ambos son gradient boosting de árboles. XGBoost regulariza fuertemente cada árbol (max_depth, min_child_weight, subsample, colsample_bytree, reg_lambda). LightGBM usa histogramas y hojas (num_leaves, min_child_samples) para ser muy rápido.

En este dataset necesitan frecuencia/target encoding para seller_id y category_id; esa compresión pierde interacciones finas que CatBoost conserva.

| familia | mejor ronda | accuracy |
|---|---:|---:|
| XGBoost | 14 | 0,897143 |
| LightGBM | 12 | 0,904571 |

**Regla práctica:** aumentar profundidad no ayudó; mejoró más la representación de categorías que la complejidad del árbol.


## Modelo 4 — Random Forest / ExtraTrees y Modelo 5 — lineales

**Random Forest** promedia muchos árboles entrenados con muestras y variables aleatorias: estable, interpretable por importancia, pero menos eficiente para categorías raras. **ExtraTrees** aleatoriza también el umbral de corte: reduce correlación entre árboles y puede ser buen segundo modelo.

**LogisticRegression / LinearSVC** optimizan un hiperplano sobre TF-IDF. C controla el compromiso margen/regularización. Son rápidos y buenos para palabras o caracteres, pero no modelan bien reglas conjuntas de vendedor, precio y logística.

En el holdout: RF tabular histórico 0,8719; logística TF-IDF 0,8355; el mejor lineal no compitió con CatBoost.


## Redes neuronales y cinco extensiones

Un MLP aplica capas densas: z = W x + b, activación ReLU y una salida sigmoide/softmax. batch_size, learning_rate, dropout, weight_decay y early_stopping controlan el entrenamiento. En estas pruebas MLP obtuvo 0,887–0,895: las categorías de alta cardinalidad y reglas irregulares favorecen árboles.

Las cinco extensiones a implementar en un ciclo aislado son: ExtraTrees, HistGradientBoosting, soft-voting/stacking OOF, TF-IDF de caracteres y una red tabular con embeddings/atención (TabNet o FT-Transformer). Cada una debe evaluarse con el mismo holdout y con un segundo fold de control.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
results = pd.read_csv(Path('results') / 'experimental_rounds_15.csv')
display(results.sort_values('accuracy', ascending=False).head(10))
ax = results.sort_values('accuracy').plot.barh(x='model', y='accuracy', figsize=(11,7), legend=False, color='#4472C4')
ax.axvline(.95, color='#C00000', linestyle='--', label='meta 0.95')
ax.set_xlabel('Accuracy en holdout agrupado'); ax.set_ylabel('Ronda / modelo'); ax.legend(); plt.tight_layout()


## Selección y submission

El modelo ganador local es la ronda 9. Para Kaggle, el helper genera ID como la posición 1-based del archivo de test y conserva condition como segundo campo. Esto es importante: el id MLA del JSONL no coincide con el identificador de evaluación de esta competencia.

~~~python
from src.ap2026q2_common import write_submission
submission = write_submission('catboost_advanced_svd', predicciones, test_ids=None)
print(submission)
~~~

Validaciones obligatorias antes de subir: 30.000 filas, header exacto ID,condition, IDs 1..30000, sin nulos y solo valores new/used.


## Conclusión

La evidencia favorece CatBoost enriquecido. La meta 0,95 no se alcanzó en 15 configuraciones/candidatos; el salto restante no parece venir de más profundidad, sino de mejorar generalización de texto y categorías raras, y de un stacking out-of-fold que combine errores realmente distintos.
